# 06. Real Somax Collaboration

This notebook uses the actual `somax` Python package (not a mock) to:
1) build a runtime corpus from MIDI,
2) generate a collaboration phrase with Somax runtime `Player`,
3) export both MIDI and WAV outputs.

Runtime targets:
- Colab standalone (auto bootstrap)
- server mode (preinstalled dependencies)

Outputs:
- `artifacts/collab_somax/session_*.mid`
- `artifacts/collab_somax/session_*.wav`
- `artifacts/collab_somax/session_report.json`


In [ ]:
from pathlib import Path
import json
import os
import random
import subprocess
import sys
from dataclasses import dataclass

import numpy as np
from mido import MidiFile, MidiTrack, Message, MetaMessage, bpm2tempo, second2tick
import matplotlib.pyplot as plt
import IPython.display as ipd

# ------------------------------------------------------------
# Runtime bootstrap (Colab + Server)
# ------------------------------------------------------------
IS_COLAB = 'google.colab' in sys.modules
RUN_MODE = os.environ.get('PIANOKIT_RUN_MODE', 'colab' if IS_COLAB else 'server')
REPO_URL = os.environ.get('PIANOKIT_REPO_URL', 'https://github.com/jhbae/pianokit.git')

if IS_COLAB and RUN_MODE == 'colab' and not Path('assets').exists():
    repo_dir = Path('/content/pianokit')
    if not repo_dir.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(repo_dir)], check=True)
    os.chdir(str(repo_dir))
    print(f'[bootstrap] changed cwd to {repo_dir}')

if IS_COLAB and RUN_MODE == 'colab':
    subprocess.run(['apt-get', 'update', '-y'], check=False)
    subprocess.run(['apt-get', 'install', '-y', 'fluidsynth', 'fluid-soundfont-gm'], check=False)
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'somax', 'mido', 'numpy==1.23.5', 'matplotlib', 'ipython'
    ], check=False)

ARTIFACTS = Path(os.environ.get('PIANOKIT_ARTIFACTS_DIR', 'artifacts'))
ASSETS = Path(os.environ.get('PIANOKIT_ASSETS_DIR', 'assets'))
SOMAX_DIR = ARTIFACTS / 'collab_somax'
SOMAX_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = int(os.environ.get('PIANOKIT_RANDOM_SEED', '42'))
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

default_soundfont = '/usr/share/sounds/sf2/FluidR3_GM.sf2'
SOUNDFONT_PATH = Path(os.environ.get('PIANOKIT_SOUNDFONT_PATH', default_soundfont))
if not SOUNDFONT_PATH.exists():
    raise FileNotFoundError(
        f'SoundFont not found: {SOUNDFONT_PATH}. '
        'Set PIANOKIT_SOUNDFONT_PATH or install fluid-soundfont-gm.'
    )

from somax.corpus_builder.corpus_builder import CorpusBuilder
from somax.runtime.player import Player

pieces = {
    'satie': {
        'midi': ARTIFACTS / 'midi' / 'satie.mid',
        'analysis': ARTIFACTS / 'analysis' / 'satie_analysis.json',
        'audio': ASSETS / 'satie_gymnopedie_no1.wav',
    },
    'prokofiev': {
        'midi': ARTIFACTS / 'midi' / 'prokofiev.mid',
        'analysis': ARTIFACTS / 'analysis' / 'prokofiev_analysis.json',
        'audio': ASSETS / 'prokofiev_toccata_op11.wav',
    },
}

print(f'Run mode: {RUN_MODE} (colab={IS_COLAB})')
print('Somax package loaded successfully.')
for k, v in pieces.items():
    print(k, 'midi', 'OK' if v['midi'].exists() else 'MISSING', 'analysis', 'OK' if v['analysis'].exists() else 'MISSING')


In [ ]:
@dataclass
class SomaxConfig:
    piece: str = "satie"
    n_events: int = 64
    tempo_bpm: float = 76.0
    session_tag: str = "real_somax_v1"

cfg = SomaxConfig()
print(cfg)


In [ ]:
# Preflight check only (main generation runs in next cell).
spec = pieces[cfg.piece]
if not spec['midi'].exists():
    raise FileNotFoundError(
        f"Missing MIDI input: {spec['midi']}. Run NB01 first or provide artifacts/midi files."
    )

print('Preflight OK:', spec['midi'])


In [ ]:
spec = pieces[cfg.piece]
if not spec['midi'].exists():
    raise FileNotFoundError(f"Missing MIDI input: {spec['midi']}. Run NB01 first.")

builder = CorpusBuilder()
corpus = builder.build(str(spec['midi']), corpus_name=f"{cfg.piece}_runtime")
print('Corpus events:', len(corpus.events))

player = Player('pianokit_player', corpus=None)
player.read_corpus(corpus)
print('Somax player ready')

generated_notes = []
generated_event_indices = []
current_time = 0.0
for step in range(cfg.n_events):
    idx = 0 if step == 0 else random.randint(0, len(corpus.events) - 1)
    generated_event_indices.append(idx)
    player.force_jump(idx)
    event_and_transform = player.new_event(scheduler_time=current_time, tempo=cfg.tempo_bpm)
    if event_and_transform is None:
        continue
    event, _ = event_and_transform
    for n in event.notes:
        generated_notes.append({
            'pitch': int(n.pitch),
            'velocity': int(max(1, min(127, n.velocity))),
            'start': float(current_time + n.onset),
            'end': float(current_time + n.onset + max(0.03, n.duration)),
        })
    current_time += float(max(0.05, event.duration))

generated_notes = sorted(generated_notes, key=lambda x: x['start'])
session_id = f"{cfg.piece}_{cfg.session_tag}"
out_midi = SOMAX_DIR / f"session_{session_id}.mid"

mid = MidiFile(ticks_per_beat=480)
track = MidiTrack()
mid.tracks.append(track)
tempo = bpm2tempo(cfg.tempo_bpm)
track.append(MetaMessage('set_tempo', tempo=tempo, time=0))

events = []
for n in generated_notes:
    events.append((n['start'], 'on', n['pitch'], n['velocity']))
    events.append((n['end'], 'off', n['pitch'], 0))
events.sort(key=lambda x: (x[0], 0 if x[1] == 'off' else 1))

cursor = 0.0
for t, kind, pitch, vel in events:
    dt = second2tick(max(0.0, t - cursor), mid.ticks_per_beat, tempo)
    if kind == 'on':
        track.append(Message('note_on', note=int(pitch), velocity=int(vel), time=int(round(dt))))
    else:
        track.append(Message('note_off', note=int(pitch), velocity=0, time=int(round(dt))))
    cursor = t

mid.save(out_midi)

out_wav = SOMAX_DIR / f"session_{session_id}.wav"
cmd = [
    'fluidsynth', '-ni', str(SOUNDFONT_PATH), str(out_midi), '-F', str(out_wav), '-r', '44100'
]
proc = subprocess.run(cmd, capture_output=True, text=True)
if proc.returncode != 0:
    print(proc.stdout)
    print(proc.stderr)
    raise RuntimeError('fluidsynth render failed')

report = {
    'session_id': session_id,
    'piece': cfg.piece,
    'somax_runtime': 'somax python package',
    'source_midi': str(spec['midi']),
    'source_audio': str(spec['audio']),
    'output_midi': str(out_midi),
    'output_wav': str(out_wav),
    'tempo_map': [{'time': 0.0, 'tempo_bpm': cfg.tempo_bpm}],
    'phrase_boundaries': [{'phrase_id': 'phrase_01', 'start_time': 0.0, 'end_time': float(current_time)}],
    'agent_actions': [
        {'agent': 'SomaxPlayer', 'action': 'force_jump_sequence', 'reason': 'real runtime event retrieval'}
    ],
    'style_profile': 'warm_legato',
    'render_version': 'somax-real-v1',
    'generated_event_indices': generated_event_indices,
}
report_path = SOMAX_DIR / 'session_report.json'
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2))

print('Saved MIDI:', out_midi)
print('Saved WAV:', out_wav)
print('Saved report:', report_path)
print('Generated notes:', len(generated_notes))


In [ ]:
times=[]
pitches=[]
for n in generated_notes:
    times.extend([n['start'], n['end'], np.nan])
    pitches.extend([n['pitch'], n['pitch'], np.nan])

plt.figure(figsize=(14,4))
plt.plot(times, pitches, linewidth=1.2)
plt.title('Somax real runtime output piano-roll trace')
plt.xlabel('Time (s)')
plt.ylabel('Pitch')
plt.grid(alpha=0.2)
plt.show()

print('Somax generated audio preview')
display(ipd.Audio(filename=str(out_wav)))
